# Naive Bayes — Distribution Estimator Only

In the HGBC variant, Naive Bayes is **not** the base learner. It is used solely for estimating missing feature-value distributions $P(F_{i,j} = v_k)$, which the SEU framework needs to compute expected utility.

The base learner (HGBC) handles classification — see `seu.ipynb`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath(".."), "src"))

import numpy as np

## Class Definition

In [ ]:
class NaiveBayesCategorical:
    """Naive Bayes for categorical features with missing-value support."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        n, d = X.shape
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.d_ = d

        class_counts = np.array([(y == c).sum() for c in self.classes_], dtype=float)
        self.log_prior_ = np.log(class_counts / class_counts.sum())

        self.feature_values_ = []
        self.log_likelihood_ = []
        self.val_to_idx_ = []

        for j in range(d):
            col = X[:, j]
            observed = ~np.isnan(col)
            vals = np.unique(col[observed]).astype(int)
            self.feature_values_.append(vals)
            n_vals = len(vals)
            val_to_idx = {v: i for i, v in enumerate(vals)}
            self.val_to_idx_.append(val_to_idx)

            counts = np.full((self.n_classes_, n_vals), self.alpha)
            if observed.any():
                obs_col = col[observed].astype(int)
                obs_y = y[observed]
                for ci, c in enumerate(self.classes_):
                    class_mask = obs_y == c
                    if class_mask.any():
                        class_vals = obs_col[class_mask]
                        for v, idx in val_to_idx.items():
                            counts[ci, idx] += (class_vals == v).sum()

            log_probs = np.log(counts / counts.sum(axis=1, keepdims=True))
            self.log_likelihood_.append(log_probs)

        self._build_lookup()
        return self

    def _build_lookup(self):
        self._max_val = max(v.max() for v in self.feature_values_ if len(v) > 0) + 1
        self._log_lk_dense = np.zeros((self.d_, self.n_classes_, self._max_val))
        self._val_known = np.zeros((self.d_, self._max_val), dtype=bool)
        for j in range(self.d_):
            log_probs = self.log_likelihood_[j]
            for v, idx in self.val_to_idx_[j].items():
                self._log_lk_dense[j, :, v] = log_probs[:, idx]
                self._val_known[j, v] = True

## Feature Value Probability Estimator

$$P(F=v \mid \text{obs}) = \sum_c P(F=v \mid C=c) \cdot P(C=c \mid \text{obs})$$

This is the **only function used by the SEU framework** from this class.

In [ ]:
def feature_value_proba(self, X_row, feature_idx):
    log_prob = self.log_prior_.copy()
    for j in range(self.d_):
        if j == feature_idx:
            continue
        v = X_row[j]
        if not np.isnan(v):
            vi = int(v)
            if vi < self._max_val and self._val_known[j, vi]:
                log_prob += self._log_lk_dense[j, :, vi]

    log_prob -= log_prob.max()
    post_class = np.exp(log_prob)
    post_class /= post_class.sum()

    vals = self.feature_values_[feature_idx]
    cond_probs = np.exp(self.log_likelihood_[feature_idx])
    marginal = cond_probs.T @ post_class
    marginal /= marginal.sum()
    return vals, marginal

NaiveBayesCategorical.feature_value_proba = feature_value_proba

## Demo

In [ ]:
import pandas as pd
from data_utils import load_breast_cancer_data, apply_mask

X_full, y = load_breast_cancer_data()

rng = np.random.default_rng(42)
X_masked_df, mask = apply_mask(pd.DataFrame(X_full), 0.50, rng)
X_masked = X_masked_df.values

rng2 = np.random.default_rng(0)
idx = rng2.permutation(len(y))
split = int(0.8 * len(y))
train_idx, val_idx = idx[:split], idx[split:]

nb = NaiveBayesCategorical()
nb.fit(X_masked[train_idx], y[train_idx])

vals, probs = nb.feature_value_proba(X_masked[0], feature_idx=0)
print(f"P(feature 0 = v | obs) for instance 0:")
for v, p in zip(vals, probs):
    print(f"  bin {v}: {p:.3f}")